# Part 0. Import libraries and data

In [ ]:
import pandas as pd
import numpy as np
from numpy import linalg as la
import math
from collections import defaultdict, Counter
from array import array
import ast
from nltk.stem import PorterStemmer
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mvelasco/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [ ]:
filename = "../../data/index_info.csv"
index_info = pd.read_csv(filename)

# List of fields, excluding 'pid': 'title', 'description', etc.
fields = index_info.columns.to_list()
fields.remove('pid')

# Parse fields as lists of terms instead of plain strings
for f in fields:
    index_info[f] = index_info[f].apply(ast.literal_eval)

# Sort by PID
index_info = index_info.sort_values('pid').reset_index(drop=True)

index_info.head(3)

,pid,title,description,brand,category,sub_category,seller,product_details
0,4hWtdShnw%3D%3D,"[men, regular, fit, solid, casual, shirt]",[],"[crock, cl]","[cloth, accessori]",[topwear],[crockscl],"[1, c328, 2, 38, regular, pure, cotton, full, ..."
1,ARME8P6GFSXGE9AW,"[welwear, suncool1008, nylon, arm, warmer, mul...","[tatto, sleev, protect, uv, raditaion, also, p...",[],"[cloth, accessori]","[winter, wear]","[navkar, craft]","[2, 2, arm, warmer, welwear, suncool1008, brow..."
2,ARME8P6GJQ8PZBE6,"[welwear, suncool1004, nylon, arm, warmer, mul...","[tatto, sleev, protect, uv, raditaion, also, p...",[],"[cloth, accessori]","[winter, wear]","[navkar, craft]","[1, 1, arm, warmer, welwear, suncool1004, blue..."


# Part 1: Indexing

## 1. Index creation

In [ ]:
def create_index(index_df: pd.DataFrame) -> defaultdict:  
    """
    Implement the inverted index
    
    :param index_df: DataFrame storing pre-processed product information 
    :returns index: the inverted index (implemented through a nested Python dictionary) containing terms and fields as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    """
    index = defaultdict(dict)

    # Iterate over each row (products)
    for i, row in index_df.iterrows():
        pid = row['pid'] # Product's ID

        # Build index only for this product and field
        # E.g. ['solid', 'women'] --> 'solid': {'title': [pid, [0]]}, 'women': {'title': [pid, [1]]}
        current_product_index = {}

        # Iterate over fields in this row: title, description, brand, ...
        for field in fields:

            terms = row[field] # List of preprocessed terms

            for position, term in enumerate(terms): # terms contains page_title + page_text. Loop over all terms
                if term in current_product_index.keys():

                    if field in current_product_index[term].keys():
                        # if the term is already in the current_product_index for the current page (current_page_current_product_index)
                        # append the position to the corresponding list
                        current_product_index[term][field][1].append(position)
                    else:
                        current_product_index[term][field] = [pid, array('I',[position])]              
                
                else:
                    # term is not in the dictionary yet
                    current_product_index[term] = {field: [pid, array('I',[position])]} #'I' indicates unsigned int (int in Python)}
                        
        # Merge the current product index with the main index
        for term, field_dict in current_product_index.items():
            for field, posting_page in field_dict.items():
                index[term].setdefault(field, []).append(posting_page)

    return index

index = create_index(index_info)

## 2. Test queries

Our extremely creative (😉) test queries will be:
1. `men's cotton checkered shirt`
2. `multicolor women casual t-shirt`
3. `black beat sports shoes`
4. `animal print jacket for women`
5. `grey hoodie with pockets`

In [ ]:
# Preprocess the text of the description column
def build_terms(text):
    """
    Preprocess text removing stop words, stemming,
    transforming in lowercase and return the tokens of the text.

    :param text: to be preprocessed
    :returns: a list of tokens corresponding to the input text after the preprocessing
    """

    stemmer = PorterStemmer()
    tokenizer = RegexpTokenizer(r'\w+')             # keep only words & numbers, drop punctuation
    stop_words = set(stopwords.words("english"))
    for w in ['not', 'no']:
        stop_words.remove(w)                        # keep 'no' and 'not' to preserve meaning of sentences like "shirt with no sleeves"
    text= text.lower()                              # transform in lowercase
    text=  tokenizer.tokenize(text)                 # tokenize the text to get a list of terms
    text=[t for t in text if t not in stop_words]   # eliminate the stopwords
    text=[stemmer.stem(t) for t in text]            # perform stemming
    return text

In [ ]:
def search(query, index):
    """
    The output is the list of documents that contain ALL of the query terms.
    So, we will get the list of documents for each query term, and take the intes of them.
    """
    query = build_terms(query)

    for i, term in enumerate(query):

        # term is not in index, there is no product with this query term
        if term not in index.keys():
            return []

        # store in term_pids the ids of the products that contain "term" in any field
        term_pids = set()
        for field in index[term].keys():
            for posting in index[term][field]:
                pid = posting[0]
                term_pids.add(pid)
        
        # docs = intersection of term_pids
        if i == 0:
            docs = term_pids
        else:
            docs = docs.intersection(term_pids)
            
    docs = list(docs)
    return docs

In [ ]:
matches_cotton = search("cotton", index)
matches_pants = search("pants", index)
matches_both = search("cotton and pants", index)

set(matches_cotton) & set(matches_pants) == set(matches_both)

True

In [ ]:
# TODO: search for test queries using "search" function

## 3. TF-IDF algorithm

In [ ]:
def create_index_tfidf(index_df: pd.DataFrame, num_documents: int):  
    """
    Implement the inverted index and compute tf, df and idf
    
    :param index_df: DataFrame storing pre-processed product information 
    :returns index: the inverted index (implemented through a nested Python dictionary) containing terms and fields as keys and the corresponding
    list of documents where these keys appears in (and the positions) as values.
    :returns tf: normalized term frequency for each term in each document
    :returns df: number of documents each term appear in
    :returns idf: inverse document frequency of each term
    """
    index = defaultdict(dict)
    tf = defaultdict(list)  # term frequencies of terms in documents (documents in the same order as in the main index)
    df = defaultdict(int)  # document frequencies of terms in the corpus
    idf = defaultdict(float)

    # Iterate over each row (products)
    for i, row in index_df.iterrows():
        pid = row['pid'] # Product's ID

        # Build index only for this product and field
        # E.g. ['solid', 'women'] --> 'solid': {'title': [pid, [0]]}, 'women': {'title': [pid, [1]]}
        current_product_index = {}

        # Iterate over fields in this row: title, description, brand, ...
        for field in fields:

            terms = row[field] # List of preprocessed terms

            for position, term in enumerate(terms): # terms contains page_title + page_text. Loop over all terms
                if term in current_product_index.keys():

                    if field in current_product_index[term].keys():
                        # if the term is already in the current_product_index for the current page (current_page_current_product_index)
                        # append the position to the corresponding list
                        current_product_index[term][field][1].append(position)
                    else:
                        current_product_index[term][field] = [pid, array('I',[position])]              
                
                else:
                    # term is not in the dictionary yet
                    current_product_index[term] = {field: [pid, array('I',[position])]} #'I' indicates unsigned int (int in Python)}
                        
        # Normalize term frequencies
        # Compute the denominator to normalize term frequencies
        # norm is the same for all terms of a document.
        norm = 0
        for term in current_product_index.keys():
            for field, posting in current_product_index[term].items():
                # posting will contain the list of positions for current term in current document.
                # posting ==> [current_doc, [list of positions]]
                # you can use it to infer the frequency of current term.
                norm += len(posting[1]) ** 2
        norm = math.sqrt(norm)

        # calculate the tf(dividing the term frequency by the above computed norm) and df weights
        for term in current_product_index.keys():
            for field, posting in current_product_index[term].items():
                # append the tf for current term (tf = term frequency in current doc/norm)
                tf[term].append(np.round(len(posting[1])/norm, 4)) ## SEE formula (1) above
                #increment the document frequency of current term (number of documents containing the current term)
                
            df[term] += 1 # increment DF for current term
        
        
        # Merge the current product index with the main index
        for term, field_dict in current_product_index.items():
            for field, posting_page in field_dict.items():
                index[term].setdefault(field, []).append(posting_page)

    # Compute IDF following the formula (3) above. HINT: use np.log
    # Note: It is computed later after we know the df.
    for term in df:
        idf[term] = np.round(np.log(float(num_documents/df[term])), 4)

    return index, tf, df, idf

index, tf, df, idf = create_index_tfidf(index_info, num_documents=len(index_info))

In [ ]:
# For now, we give equal weights to all fields (1/7)
field_weights = np.array(len(fields)*[1/len(fields)])
field_weights

array([0.14285714, 0.14285714, 0.14285714, 0.14285714, 0.14285714,
       0.14285714, 0.14285714])

In [ ]:
# TODO: finish this part
# implement field weights (if all weights are equal the result should be the same as regular TF-IDF)

# TODO: create title_index, a dictionary where keys are product PIDS and values are the original titles

def rank_documents(terms, docs, index, idf, tf, title_index):
    """
    Perform the ranking of the results of a search based on the tf-idf weights

    Argument:
    terms -- list of query terms
    docs -- list of documents, to rank, matching the query
    index -- inverted index data structure
    idf -- inverted document frequencies
    tf -- term frequencies
    title_index -- mapping between page id and page title

    Returns:
    Print the list of ranked documents
    """

    # I'm interested only on the element of the docVector corresponding to the query terms
    # The remaining elements would became 0 when multiplied to the query_vector
    doc_vectors = defaultdict(lambda: [0] * len(terms)) # I call doc_vectors[k] for a nonexistent key k, the key-value pair (k,[0]*len(terms)) will be automatically added to the dictionary
    query_vector = [0] * len(terms)

    # compute the norm for the query tf
    query_terms_count = Counter(terms)  # get the frequency of each term in the query.
    # Example: collections.Counter(["hello","hello","world"]) --> Counter({'hello': 2, 'world': 1})

    query_norm = la.norm(list(query_terms_count.values()))

    for termIndex, term in enumerate(terms):  #termIndex is the index of the term in the query
        if term not in index:
            continue

        ## Compute tf*idf(normalize TF as done with documents)
        query_vector[termIndex]=query_terms_count[term]/query_norm * idf[term] #query_vector[0] corresponds to the first term in the query

        # Generate doc_vectors for matching docs
        for doc_index, (doc, postings) in enumerate(index[term]):
            # Example of [doc_index, (doc, postings)]
            # 0 (26, array('I', [1, 4, 12, 15, 22, 28, 32, 43, 51, 68, 333, 337]))
            # 1 (33, array('I', [26, 33, 57, 71, 87, 104, 109]))
            # term is in doc 26 in positions 1,4, .....
            # term is in doc 33 in positions 26,33, .....

            #tf[term][0] will contain the tf of the term "term" in the doc 26
            if doc in docs: #if the odcument is in the list of documents retrieved (matching the query)
                doc_vectors[doc][termIndex] = tf[term][doc_index] * idf[term]  # TODO: check if multiply for idf

    # Calculate the score of each doc
    # compute the cosine similarity between queyVector and each docVector:
    # HINT: you can use the dot product because in case of normalized vectors it corresponds to the cosine similarity
    # see np.dot

    doc_scores=[[np.dot(curDocVec, query_vector), doc] for doc, curDocVec in doc_vectors.items() ]
    doc_scores.sort(reverse=True)
    result_docs = [x[1] for x in doc_scores]

    return result_docs

In [ ]:
def search_tf_idf(query, index):
    """
    output is the list of documents that contain any of the query terms.
    So, we will get the list of documents for each query term, and take the union of them.
    """
    query = build_terms(query)
    docs = set()
    for term in query:
        try:
            # store in term_docs the ids of the docs that contain "term"
            term_docs=[posting[0] for posting in index[term]]

            # docs = docs Union term_docs
            docs = docs.union(term_docs)
        except:
            #term is not in index
            pass
    docs = list(docs)
    ranked_docs = rank_documents(query, docs, index, idf, tf, title_index)
    return ranked_docs

# Part 2

In [ ]:
#TODO: copy functions from Lab2